In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np

from convert import convert_conllu_list_to_csv
from utils import *

In [2]:
manifest = [
    ("ud",      "gold", "../datasets/ud/test.conllu"),
    ("ud",      "pred", "../out/ud/test.pred.conllu"),
    ("ud-new",  "gold", "../datasets/ud-new/test.conllu"),
    ("ud-new",  "pred", "../out/ud-new/test.pred.conllu"),
    ("ud-old",  "gold", "../datasets/ud-old/test.conllu"),
    ("ud-old",  "pred", "../out/ud-old/test.pred.conllu"),

    ("str",     "gold", "../datasets/str/test.conllu"),
    ("str",     "pred", "../out/str/test.pred.conllu"),
    ("str-new", "gold", "../datasets/str-new/test.conllu"),
    ("str-new", "pred", "../out/str-new/test.pred.conllu"),
    ("str-old", "gold", "../datasets/str-old/test.conllu"),
    ("str-old", "pred", "../out/str-old/test.pred.conllu"),
]

inputs = [p for _, _, p in manifest]
csvs = convert_conllu_list_to_csv(inputs, "csvs")

csv_index = {}
for (split, role, _), csv in zip(manifest, csvs):
    csv_index.setdefault(split, {})[role] = Path(csv)

DATA = {split: load_pair(paths["gold"], paths["pred"]) for split, paths in csv_index.items()}

pairs = {
    "full": ("str",     "ud"),
    "old":  ("str-old", "ud-old"),
    "new":  ("str-new", "ud-new"),
}


In [3]:
OBLIGATORY_FEATS = ["Number", "Gender", "Case", "Animacy", "Aspect", "Mood", "Person", "VerbForm", "Degree"]
STR_KEY_FOR_UD_FEAT = {
    "Number":   "Число", 
    "Gender":   "Род", 
    "Case":     "Падеж", 
    "Animacy":  "Одуш", 
    "Aspect":   "Вид", 
    "Mood":     "Накл", 
    "Person":   "Лицо",
    "VerbForm": "Форма",
    "Degree":   "СтепСравн",
}
STR_DROP_VALUES = {
    "Case": {"ПАРТ", "МЕСТ", "ПРЕДЛ", "НЕСМЕН"},
}

In [4]:
# Cell 5 — parsers + normalization helpers

def _norm(v):
    if v is None:
        return "∅"
    v = str(v).strip()
    return v if v else "∅"


def parse_str_feats_ru(feats: str) -> dict:
    """
    'Вид=НЕСОВ|Время=ПРОШ|...' -> {'Вид':'НЕСОВ', ...}
    """
    if feats is None:
        return {}
    feats = str(feats).strip()
    if feats in ("", "_", "nan"):
        return {}
    out = {}
    for part in feats.split("|"):
        part = part.strip()
        if not part or "=" not in part:
            continue
        k, v = part.split("=", 1)
        out[k.strip()] = v.strip()
    return out


def extract_feat_value_raw(feats_str: str, ud_feat: str, source: str) -> str:
    if feats_str is None:
        return "∅"

    if source == "ud":
        d = feats_to_dict(feats_str)
        return _norm(d.get(ud_feat))

    # source == "str": русские ключи/значения — возвращаем как есть
    d_ru = parse_str_feats_ru(feats_str)
    ru_key = STR_KEY_FOR_UD_FEAT.get(ud_feat)
    if ru_key is None:
        return "∅"
    return _norm(d_ru.get(ru_key))



def add_feat_columns_raw(df: pd.DataFrame, ud_feat: str, source: str) -> pd.DataFrame:
    """
    Adds feat_g / feat_p columns in UD-normalized space.
    Requires df has feats_g and feats_p (as produced by load_pair()).
    """
    out = df.copy()
    out["feat_g"] = out["feats_g"].map(lambda s: extract_feat_value_raw(s, ud_feat, source))
    out["feat_p"] = out["feats_p"].map(lambda s: extract_feat_value_raw(s, ud_feat, source))
    return out


In [5]:
# Cell 6 — core: find most frequent mismatches (STR correct, UD wrong)

def find_most_frequent_feat_errors_raw(str_df: pd.DataFrame, ud_df: pd.DataFrame, ud_feat: str) -> pd.DataFrame:
    s = add_feat_columns_raw(str_df, ud_feat, source="str").copy()
    u = add_feat_columns_raw(ud_df,  ud_feat, source="ud").copy()

    # rename to keep after merge
    s = s.rename(columns={"feat_g": "feat_g_str", "feat_p": "feat_p_str"})
    u = u.rename(columns={"feat_g": "feat_g_ud",  "feat_p": "feat_p_ud"})

    merged = s.reset_index().merge(u.reset_index(), on=["sent_id", "id"], suffixes=("_str", "_ud"))

    present = (merged["feat_g_str"] != "∅") & (merged["feat_g_ud"] != "∅")
    mismatch = present & (merged["feat_g_str"] == merged["feat_p_str"]) & (merged["feat_g_ud"] != merged["feat_p_ud"])

    m = merged[mismatch].copy()

    freq = (
        m.groupby(["feat_g_str", "feat_g_ud", "feat_p_ud"], as_index=False)
         .size()
         .rename(columns={"size": "count"})
         .sort_values("count", ascending=False)
    )
    freq["feat"] = ud_feat
    return freq


def build_feat_error_matrix(freq_by_split: dict, min_total_count: int = 1) -> pd.DataFrame:
    frames = []
    for split_name, df in freq_by_split.items():
        tmp = df.copy()
        tmp["split"] = split_name
        frames.append(tmp)

    if not frames:
        return pd.DataFrame()

    all_err = pd.concat(frames, ignore_index=True)

    mat = all_err.pivot_table(
        index=["feat", "feat_g_str", "feat_g_ud", "feat_p_ud"],
        columns="split",
        values="count",
        fill_value=0,
        aggfunc="sum",
    )

    mat["total"] = mat.sum(axis=1)
    mat = mat[mat["total"] >= min_total_count].sort_values("total", ascending=False)

    cols = [c for c in ["full", "old", "new"] if c in mat.columns] + [c for c in mat.columns if c not in ("full","old","new")]
    return mat[cols]


In [6]:
# Cell 7 — collect ALL mismatches for browse-like saving

def get_all_feat_errors_raw(str_df: pd.DataFrame, ud_df: pd.DataFrame, ud_feat: str) -> pd.DataFrame:
    s = add_feat_columns_raw(str_df, ud_feat, source="str").copy()
    u = add_feat_columns_raw(ud_df,  ud_feat, source="ud").copy()

    s = s.rename(columns={"feat_g": "feat_g_str", "feat_p": "feat_p_str"})
    u = u.rename(columns={"feat_g": "feat_g_ud",  "feat_p": "feat_p_ud"})

    merged = s.reset_index().merge(u.reset_index(), on=["sent_id", "id"], suffixes=("_str", "_ud"))

    present = (merged["feat_g_str"] != "∅") & (merged["feat_g_ud"] != "∅")
    mismatch = present & (merged["feat_g_str"] == merged["feat_p_str"]) & (merged["feat_g_ud"] != merged["feat_p_ud"])

    out = merged[mismatch].copy()
    out["feat_name"] = ud_feat
    out = out.sort_values(["sent_id", "id"]).reset_index(drop=True)
    return out


In [7]:
# Cell 8 — browse_upos_errors-like dump to txt (non-interactive)

def _browse_like_lines(df: pd.DataFrame, title: str) -> list[str]:
    lines = []
    lines.append(title)
    lines.append(f"Total examples: {len(df)}")

    # choose which sentence text to show
    text_cols = [c for c in ["text_ud", "text_str", "text"] if c in df.columns]
    form_col = "form_str" if "form_str" in df.columns else ("form_ud" if "form_ud" in df.columns else ("form" if "form" in df.columns else None))

    for i, row in df.iterrows():
        lines.append("=" * 90)
        lines.append(f"#{i}  sent_id={row.get('sent_id')}  id={row.get('id')}")
        lines.append("-" * 90)

        if text_cols:
            lines.append("Sentence:")
            lines.append(str(row.get(text_cols[0])))
        else:
            lines.append("Sentence: (no text column)")

        if form_col and "sent_id" in df.columns:
            toks = df[df["sent_id"] == row["sent_id"]][form_col].tolist()
            lines.append("Tokens:")
            lines.append(" ".join(map(str, toks)))

        lines.append("-" * 90)
        if form_col:
            lines.append(f"Token: {row.get(form_col)!r}")

        # show raw feats if present
        for col in ["feats_g_str", "feats_p_str", "feats_g_ud", "feats_p_ud", "feats_g", "feats_p"]:
            if col in df.columns:
                lines.append(f"{col}: {row.get(col)}")

        lines.append("Compared feature:")
        lines.append(f"  feat: {row.get('feat_name')}")
        lines.append(f"  STR gold: {row.get('feat_g_str')}   STR pred: {row.get('feat_p_str')}")
        lines.append(f"  UD  gold: {row.get('feat_g_ud')}    UD  pred: {row.get('feat_p_ud')}")

    return lines


def save_browse_like(df: pd.DataFrame, path: Path, title: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(_browse_like_lines(df, title)), encoding="utf-8")


In [8]:
# Cell 9 — export: tables as PNG like your UPOS workflow + examples as browse-like txt

# If needed:
# %pip install -q dataframe_image
import dataframe_image as dfi

OUT_DIR = Path("morph_common")
TABLES_PNG_DIR = OUT_DIR / "tables_png"
TABLES_CSV_DIR = OUT_DIR / "tables_csv"
EXAMPLES_DIR = OUT_DIR / "examples_txt"

TABLES_PNG_DIR.mkdir(parents=True, exist_ok=True)
TABLES_CSV_DIR.mkdir(parents=True, exist_ok=True)
EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

for feat in OBLIGATORY_FEATS:
    # 1) compute per split
    freq_by_split = {}
    for split_name, (split_str, split_ud) in pairs.items():
        freq_by_split[split_name] = find_most_frequent_feat_errors_raw(DATA[split_str], DATA[split_ud], feat)

    # 2) matrix table
    mat = build_feat_error_matrix(freq_by_split, min_total_count=1)

    # save CSV (backup)
    mat.reset_index().to_csv(TABLES_CSV_DIR / f"{feat}.error_matrix.csv", index=False, encoding="utf-8")

    # save PNG
    if len(mat) > 0:
        dfi.export(
            mat,
            TABLES_PNG_DIR / f"{feat}.error_matrix.png",
            table_conversion="matplotlib",
            dpi=300,
            fontsize=12,
        )
    else:
        # still create an empty placeholder csv already done; skip png
        pass

    # 3) save ALL examples (browse-like) per split
    for split_name, (split_str, split_ud) in pairs.items():
        all_ex = get_all_feat_errors_raw(DATA[split_str], DATA[split_ud], feat)
        title = f"{feat} / split={split_name}  (STR correct, UD wrong)"
        save_browse_like(all_ex, EXAMPLES_DIR / feat / f"{split_name}.ALL.txt", title=title)

print("Saved to:", OUT_DIR.resolve())


Saved to: C:\Users\danit\Desktop\Drive\Code\Study\SynTagRus-UD-Comparison\morph\morph_common
